# 00 — Ollama en Colab (setup canónico)

Este notebook instala **Ollama** en la instancia de Colab y baja los modelos que usan los demás notebooks. Es el lugar canónico con la explicación completa; los notebooks 04–08 llevan la versión compacta e idempotente de estas mismas celdas, así que cada uno puede correrse solo.

**Antes de correrlo:** activa el runtime con GPU (Runtime > Change runtime type > T4 GPU) y verifica con `!nvidia-smi`.

**Diseño:** los modelos intermedios (lector RAG, ejecutivo MEMO, juez RAGAS, LLM del agente) corren vía Ollama en la propia instancia: un solo servidor, cambio de modelo con un parámetro, sin cargar pesos en cada notebook. El entrenamiento MEMO (notebook 03) sigue siendo Hugging Face + LoRA — entrenar no se puede hacer vía Ollama.

**Fuente de los comandos:** guías publicadas de Ollama en Colab (apt `zstd` + `install.sh` + `ollama serve` en background + pull de modelos) y verificación propia de que Colab no trae `zstd` y el instalador lo exige.

In [ ]:
%%bash
nvidia-smi
python3 -c "import torch; print('torch', torch.__version__, '| CUDA disponible:', torch.cuda.is_available())" 2>/dev/null || echo "torch aun no instalado (se instala abajo)"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive montado en /content/drive")

In [ ]:
# --- Parametros (editalos aqui arriba) ---
DRIVE_ROOT = "/content/drive/MyDrive/femsa-rag"   # carpeta en Drive donde vive todo
REPO_URL = "https://github.com/czammar/femsa-rag-graph-memo"
REPO = DRIVE_ROOT + "/femsa-rag-graph-memo"

In [ ]:
import os, subprocess

os.makedirs(DRIVE_ROOT, exist_ok=True)
# El cache de Hugging Face vive en Drive: los pesos descargados
# sobreviven al reinicio de la sesion (la VM de Colab es efimera).
os.environ["HF_HOME"] = DRIVE_ROOT + "/.cache/huggingface"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

if os.path.isdir(REPO + "/.git"):
    print("repo ya existe, actualizando...")
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=False)
else:
    print("clonando el repo...")
    subprocess.run(["git", "clone", REPO_URL, REPO], check=True)
print("repo en:", REPO)
print("HF_HOME:", os.environ["HF_HOME"])

## 1. Instalar Ollama

Colab no trae `zstd` y el instalador de Ollama lo exige: primero `apt-get` con `zstd` (varias guías añaden también `pciutils` para detectar la GPU) y luego el `install.sh` oficial. Si el binario ya existe, no se reinstala nada.

In [ ]:
%%bash
if command -v ollama >/dev/null 2>&1; then
  echo "ollama ya instalado: $(ollama --version)"
else
  echo "instalando Ollama (primero zstd, que el instalador exige y Colab no trae)..."
  apt-get -qq update && apt-get -qq install -y zstd pciutils
  curl -fsSL https://ollama.com/install.sh | sh
  ollama --version
fi

## 2. Arrancar el servidor en segundo plano

`ollama serve` en primer plano bloquearía la celda para siempre, así que se lanza con `nohup` + `setsid` y se espera a que responda en el puerto **11434**. Si el servidor ya responde, no se arranca otro. `OLLAMA_HOST=0.0.0.0` solo hace falta si se expone con un túnel; para uso dentro del notebook basta `localhost`.

In [ ]:
import os, subprocess, time, urllib.request

def ollama_up():
    try:
        with urllib.request.urlopen("http://localhost:11434", timeout=3) as r:
            return r.status == 200
    except Exception:
        return False

if ollama_up():
    print("servidor Ollama ya responde en http://localhost:11434")
else:
    print("arrancando `ollama serve` en segundo plano...")
    out = open("ollama.log", "w")
    err = open("ollama_error.log", "w")
    subprocess.Popen(["nohup", "ollama", "serve"], stdout=out, stderr=err,
                     preexec_fn=os.setsid)
    for _ in range(60):
        if ollama_up():
            break
        time.sleep(2)
    print("Ollama responde en localhost:11434:", ollama_up())
    if not ollama_up():
        print("Revisa ollama_error.log; el servidor no levanto.")

## 3. Bajar los modelos

Tamaños en VRAM (T4, 16 GB): el **7B cómodo** (~4.7 GB en Q4_K_M), el **14B Q4 ~9 GB, ajustado** (cabe, pero deja poco margen junto a los embeddings). El 1.5B es el modelo de los runs originales en CPU (paridad); el 7B es el modelo más grande para el agente y el juez.

**Ojo:** los modelos viven en la VM (`~/.ollama/models`) y **hay que volver a bajarlos en cada sesión nueva** — la VM de Colab es efímera. Opcional: apuntar `OLLAMA_MODELS` a una ruta en Drive para persistirlos, a costa de más lentitud (Drive es más lento que el disco local):

```bash
export OLLAMA_MODELS=/content/drive/MyDrive/femsa-rag/ollama-models
```

Esa variable debe definirse **antes** de arrancar el servidor. Por defecto no se usa: bajar de nuevo es más rápido que servir desde Drive.

In [ ]:
%%bash
if ollama show "qwen2.5:1.5b" >/dev/null 2>&1; then
  echo "qwen2.5:1.5b ya descargado"
else
  echo "bajando qwen2.5:1.5b..."
  ollama pull "qwen2.5:1.5b"
fi
if ollama show "qwen2.5:7b" >/dev/null 2>&1; then
  echo "qwen2.5:7b ya descargado"
else
  echo "bajando qwen2.5:7b..."
  ollama pull "qwen2.5:7b"
fi
ollama list

In [ ]:
# Opcional: el 14B, solo si cabe (T4 16 GB: ~9 GB, ajustado).
# Quita el comentario para bajarlo.
# !ollama pull qwen2.5:14b
print("Modelos listos. El 14B es opcional: quita el comentario de la celda si lo quieres.")

## 4. Verificación: el servidor responde y el modelo contesta

In [ ]:
from langchain_ollama import ChatOllama

for _m in ["qwen2.5:1.5b", "qwen2.5:7b"]:
    _llm = ChatOllama(model=_m, base_url="http://localhost:11434", temperature=0)
    print(_m, "->", _llm.invoke("Responde con una sola palabra: ¿cuánto es 2+2?").content)
print("Ollama OK: servidor en localhost:11434, modelos 1.5b y 7b listos.")

## Qué sigue

- **Notebook 01:** setup general + extracción del informe (no necesita Ollama).
- **Notebooks 04–08:** cada uno trae su propio bloque compacto de Ollama (instalar/arrancar/pull idempotentes), así que pueden correrse solos sin haber corrido este 00 primero — pero si ya lo corriste, no descargan nada de más.

In [ ]:
import subprocess

ver = subprocess.run(["ollama", "--version"], capture_output=True, text=True).stdout.strip()
lst = subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout
manifest = f"ollama {ver}\n\nModelos bajados en esta sesion:\n{lst}"
open(REPO + "/ollama-manifest.txt", "w").write(manifest)
print(manifest)
print("manifiesto ->", REPO + "/ollama-manifest.txt")

## Guardar en Drive (final)

Todo lo que produjo este notebook vive bajo el repo clonado en Drive (`REPO`), así que sobrevive aunque la sesión de Colab se corte. Esta celda lo verifica y lo lista: si algo falta, lo dice aquí, no después.

Los modelos en sí NO se copian a Drive (4.7–9 GB cada uno; bajar de nuevo por sesión es más rápido que servirlos desde Drive). Lo que queda guardado es este manifiesto con la versión y los tags bajados.

In [ ]:
import os, shutil, time


def _human(n):
    for u in ["B", "KB", "MB", "GB"]:
        if n < 1024:
            return f"{n:.0f} {u}"
        n /= 1024
    return f"{n:.1f} GB"


def _list_dir(path):
    total, nfiles = 0, 0
    for root, _dirs, files in os.walk(path):
        for fn in sorted(files):
            p = os.path.join(root, fn)
            sz = os.path.getsize(p)
            total += sz
            nfiles += 1
            print(f"    {os.path.relpath(p, REPO)} ({_human(sz)})")
    return nfiles, total


# Logs de la sesion (viven en /content, fuera de Drive): copiarlos al repo.
for _log in ["ollama.log", "ollama_error.log", "relate.log"]:
    if os.path.isfile(_log):
        shutil.copy(_log, os.path.join(REPO, _log))
        print(f"log copiado a Drive: {_log}")

ENTRIES = [('ollama-manifest.txt', 'Manifiesto Ollama (modelos bajados)', True)]
total_bytes, missing = 0, []
for rel, label, required in ENTRIES:
    p = os.path.join(REPO, rel)
    if os.path.isdir(p):
        nf, tb = _list_dir(p)
        total_bytes += tb
        print(f"{label}: directorio, {nf} archivos, {_human(tb)}")
    elif os.path.isfile(p):
        sz = os.path.getsize(p)
        total_bytes += sz
        mt = time.strftime("%Y-%m-%d %H:%M", time.localtime(os.path.getmtime(p)))
        print(f"{label}: {rel} ({_human(sz)}, {mt})")
    else:
        status = "FALTA (obligatorio)" if required else "pendiente (opcional)"
        print(f"{label}: {status}: {rel}")
        if required:
            missing.append(rel)
print(f"\nGUARDADO EN DRIVE: {_human(total_bytes)} verificados bajo {REPO}")
if missing:
    print("OJO, faltan obligatorios:", missing)